In [ ]:
from pathlib import Path

def _sionna_repo_root():
    """Find project root when Jupyter is started from root or notebooks/."""
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "scenes" / "University.xml").is_file():
            return candidate
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ directory")


import numpy as np
from scipy.io import loadmat
import sionna
import sionna.rt
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver

mat_path = str(_sionna_repo_root() / "data" / "Beamforming_1User_Init_Opt.mat")
if not Path(mat_path).is_file():
    raise FileNotFoundError(
        "External MATLAB data is not included in this public repository. "
        "Place an authorized Beamforming_1User_Init_Opt.mat under data/."
    )
data = loadmat(mat_path, squeeze_me=True, struct_as_record=False)

h = np.asarray(data["h"]).reshape(-1)
w_init = np.asarray(data["w_init"]).reshape(-1)
w_opt = np.asarray(data["w_opt"]).reshape(-1)
info = data["info"]

fc = float(info.fc)
M = int(info.M)
bs_center = np.asarray(info.bs_center, dtype=float).reshape(-1)
ue_pos = np.asarray(info.ue_pos, dtype=float).reshape(-1)

print("fc =", fc)
print("M =", M)
print("BS position =", bs_center)
print("UE position =", ue_pos)

scene = load_scene(sionna.rt.scene.simple_street_canyon, merge_shapes=False)
scene.frequency = fc

scene.tx_array = PlanarArray(
    num_rows=M,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization="V"
)

scene.rx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization="V"
)

for name in list(scene.transmitters.keys()):
    scene.remove(name)

for name in list(scene.receivers.keys()):
    scene.remove(name)

tx = Transmitter(
    position=bs_center.tolist(),
    name="bs",
    color=[0.0, 0.0, 1.0]
)

rx = Receiver(
    position=ue_pos.tolist(),
    name="ue",
    color=[1.0, 0.0, 0.0]
)

scene.add(tx)
scene.add(rx)
tx.look_at(rx)

p_solver = PathSolver()

paths_los = p_solver(
    scene=scene,
    max_depth=0,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    refraction=False,
    synthetic_array=False,
    seed=42
)

print("Path computation done.")

a, tau = paths_los.cir(normalize_delays=False, out_type="numpy")
print("a shape  :", a.shape)
print("tau shape:", tau.shape)

In [ ]:
import numpy as np
import mitsuba as mi
from sionna.rt import RadioMapSolver

# ============================================================
# 1) Precoding vector를 Sionna 형식으로 변환
# ============================================================
def to_precoding_tuple(w):
    w = np.asarray(w).reshape(-1)
    return (
        mi.TensorXf(w.real.astype(np.float32)),
        mi.TensorXf(w.imag.astype(np.float32))
    )

prec_init = to_precoding_tuple(w_init)
prec_opt  = to_precoding_tuple(w_opt)

# ============================================================
# 2) Radio map 계산
#    - 두 경우(init/opt) 모두 동일한 plane 사용
# ============================================================
rm_solver = RadioMapSolver()

rm_center = [15.0, 0.0, 1.5]
rm_size   = [30.0, 30.0]
cell_size = (0.25, 0.25)

rm_init = rm_solver(
    scene,
    max_depth=0,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    refraction=False,
    diffraction=False,
    samples_per_tx=10**6,
    precoding_vec=prec_init,
    cell_size=cell_size,
    center=rm_center,
    size=rm_size,
    orientation=[0, 0, 0]
)

rm_opt = rm_solver(
    scene,
    max_depth=0,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    refraction=False,
    diffraction=False,
    samples_per_tx=10**6,
    precoding_vec=prec_opt,
    cell_size=cell_size,
    center=rm_center,
    size=rm_size,
    orientation=[0, 0, 0]
)

# ============================================================
# 3) display range를 각각 따로 계산
# ============================================================
pg_init = np.array(rm_init.path_gain)
pg_opt  = np.array(rm_opt.path_gain)

pg_init_db = 10*np.log10(pg_init + 1e-30)
pg_opt_db  = 10*np.log10(pg_opt + 1e-30)

print("init dB min/max:", pg_init_db.min(), pg_init_db.max())
print("opt  dB min/max:", pg_opt_db.min(), pg_opt_db.max())

dynamic_range_db = 15

rm_vmax_init = np.ceil(pg_init_db.max())
rm_vmin_init = rm_vmax_init - dynamic_range_db

rm_vmax_opt = np.ceil(pg_opt_db.max())
rm_vmin_opt = rm_vmax_opt - dynamic_range_db

print("Initial display range:")
print("rm_vmin_init =", rm_vmin_init)
print("rm_vmax_init =", rm_vmax_init)

print("Optimized display range:")
print("rm_vmin_opt =", rm_vmin_opt)
print("rm_vmax_opt =", rm_vmax_opt)


print("Initial beam preview")
scene.preview(
    radio_map=rm_init,
    rm_metric="path_gain",
    rm_db_scale=True,
    rm_vmin=rm_vmin_init,
    rm_vmax=rm_vmax_init,
    clip_at=100
)

print("Optimized beam preview")
scene.preview(
    radio_map=rm_opt,
    rm_metric="path_gain",
    rm_db_scale=True,
    rm_vmin=rm_vmin_opt,
    rm_vmax=rm_vmax_opt,
    clip_at=100
)

# ============================================================
# 6) XY beam map
#    - 여기서도 같은 color range(0 ~ -20 dB)로 표시
# ============================================================
bs = bs_center.astype(float)
ue = ue_pos.astype(float)

z_plane = 1.5
x_min, x_max = -5.0, 35.0
y_min, y_max = -20.0, 20.0
nx, ny = 201, 201

xg = np.linspace(x_min, x_max, nx)
yg = np.linspace(y_min, y_max, ny)
X, Y = np.meshgrid(xg, yg)
Z = np.full_like(X, z_plane)

def steering_vector_from_point(point_xyz, bs_xyz, M):
    v = np.asarray(point_xyz, dtype=float) - np.asarray(bs_xyz, dtype=float)
    r = np.linalg.norm(v)
    if r < 1e-12:
        return np.ones(M, dtype=complex) / np.sqrt(M)
    u = v / r
    uy = u[1]
    n = np.arange(M)
    a = np.exp(1j * np.pi * n * uy) / np.sqrt(M)
    return a

def beam_gain_at_point(point_xyz, w, bs_xyz, M):
    a = steering_vector_from_point(point_xyz, bs_xyz, M)
    return np.abs(np.vdot(w, a))**2

G_init = np.zeros_like(X, dtype=float)
G_opt  = np.zeros_like(X, dtype=float)

for i in range(ny):
    for j in range(nx):
        p = np.array([X[i, j], Y[i, j], Z[i, j]])
        G_init[i, j] = beam_gain_at_point(p, w_init, bs, M)
        G_opt[i, j]  = beam_gain_at_point(p, w_opt,  bs, M)

G_init_db = 10*np.log10(G_init / np.max(G_init) + 1e-12)
G_opt_db  = 10*np.log10(G_opt  / np.max(G_opt)  + 1e-12)

plt.figure(figsize=(12, 5))

ax1 = plt.subplot(1, 2, 1)
im1 = ax1.imshow(
    G_init_db,
    extent=[x_min, x_max, y_min, y_max],
    origin="lower",
    aspect="auto",
    vmin=-20,
    vmax=0,
    cmap="jet"
)
ax1.scatter(bs[0], bs[1], c="white", s=70, edgecolors="k", label="BS")
ax1.scatter(ue[0], ue[1], c="red", s=70, edgecolors="k", label="UE")
ax1.plot([bs[0], ue[0]], [bs[1], ue[1]], "w--", linewidth=1.5, label="LoS line")
ax1.set_title(f"XY Beam Map: Initial (z={z_plane} m)")
ax1.set_xlabel("x [m]")
ax1.set_ylabel("y [m]")
ax1.legend()
plt.colorbar(im1, ax=ax1, label="Normalized beam power [dB]")

ax2 = plt.subplot(1, 2, 2)
im2 = ax2.imshow(
    G_opt_db,
    extent=[x_min, x_max, y_min, y_max],
    origin="lower",
    aspect="auto",
    vmin=-20,
    vmax=0,
    cmap="jet"
)
ax2.scatter(bs[0], bs[1], c="white", s=70, edgecolors="k", label="BS")
ax2.scatter(ue[0], ue[1], c="red", s=70, edgecolors="k", label="UE")
ax2.plot([bs[0], ue[0]], [bs[1], ue[1]], "w--", linewidth=1.5, label="LoS line")
ax2.set_title(f"XY Beam Map: Optimized (z={z_plane} m)")
ax2.set_xlabel("x [m]")
ax2.set_ylabel("y [m]")
ax2.legend()
plt.colorbar(im2, ax=ax2, label="Normalized beam power [dB]")

plt.tight_layout()
plt.show()

In [ ]:
pg_init = np.array(rm_init.path_gain)
pg_opt  = np.array(rm_opt.path_gain)

print("Are they exactly equal? ->", np.allclose(pg_init, pg_opt))
print("Max abs difference      ->", np.max(np.abs(pg_opt - pg_init)))
print("Mean abs difference     ->", np.mean(np.abs(pg_opt - pg_init)))